IDREES
# Migrant generation (built from 3 columns)
1- What is your country of origin? (compared with the country of residence)
2- In which country are you located? (if it differs from the country of origin then -> first generation)
3- Which description fits your parents’ country of origin the best? (if a non-EU parent and living in the country of origin then-> second generation)
origin != located (currently)= 1st gen
same country + non eu parents = 2nd gen 
else -> non migrant 
# ON1-ON2-ON3 cleaned and processed 
# DVP1-DVP5 cleaned and processed 
# PF = mean(F1, F2)


In [1]:
import pandas as pd
import numpy as np

In [2]:
#(Idrees) read medeea's final combined file so our located + origin cleaning is added on top of her work (likert + parents)
df = pd.read_excel('Responses_cleaned_finalMedeea.xlsx')
df.shape

(603, 72)

Cleaning: "In which country are you located"
normalize (strip + lowercase) then map every variant to one clean country name.

In [3]:


col = 'In which country are you located?'

In [4]:
#(Idrees)check the raw values
df[col].value_counts(dropna=False)

In which country are you located?
The Netherlands    267
Germany            135
Spain               98
Greece              64
Austria              4
United Kingdom       4
Malta                3
France               3
Sweden               3
United States        2
Canada               2
Belgium              2
Morocco              2
Portugal             2
New Zealand          1
Singapore            1
South Africa         1
Poland               1
Italy                1
Hong Kong            1
Spiliotopoulos       1
Jordan               1
Brazil               1
Egypt                1
Turkey               1
NaN                  1
Name: count, dtype: int64

In [5]:
#(Idrees)number of unique values and missing count
print('Unique values:', df[col].nunique(dropna=True))
print('Missing:', df[col].isna().sum())

Unique values: 25
Missing: 1


In [6]:
#(Idrees)- fix all netherlands variants to "The Netherlands"
netherlands_variants = [
    'netherlands', 'the netherlands', 'nederland', 'niederlande', 'holland', 'nl',
    'netherland', 'the netherlans', 'netherworld',
    'amsterdam', 'amsterxam', 'uithoorn', 'netherlands, amsterdam',
]

#compare on a stripped + lowercased version so spacing and case don't matter
normalized = df[col].astype(str).str.strip().str.lower()
df.loc[normalized.isin(netherlands_variants), col] = 'The Netherlands'

#check the result
df[col].value_counts(dropna=False)
print(normalized.isin(netherlands_variants).sum(), 'rows updated to "The Netherlands"')

267 rows updated to "The Netherlands"


In [7]:
#(Idrees)-clean all the other countries the same way, using a normalized then clean-name map
#keys are stripped + lowercased, so case/spacing variants are handled automatically
country_map = {
    'germany': 'Germany', 'deutschland': 'Germany', 'german': 'Germany',
    'spain': 'Spain', 'españa': 'Spain', 'espana': 'Spain',
    'greece': 'Greece', 'greece.': 'Greece',
    'turkiye': 'Turkey', 'türkiye': 'Turkey', 'تركيا': 'Turkey',
    'egypt': 'Egypt', 'cairo': 'Egypt',
    'united kingdom': 'United Kingdom', 'uk': 'United Kingdom', 'england': 'United Kingdom',
    'usa': 'United States', 'united states': 'United States',
    'malta': 'Malta', 'france': 'France', 'sweden': 'Sweden', 'austria': 'Austria',
    'morocco': 'Morocco', 'belgium': 'Belgium', 'canada': 'Canada', 'portugal': 'Portugal',
    'brazil': 'Brazil', 'jordan': 'Jordan', 'norway': 'Norway', 'india': 'India',
    'new zealand': 'New Zealand', 'poland': 'Poland', 'south africa': 'South Africa',
    'switzerland': 'Switzerland', 'italy': 'Italy', 'hong kong': 'Hong Kong',
    'singapore': 'Singapore',
}

normalized = df[col].astype(str).str.strip().str.lower()
mapped = normalized.map(country_map)
#where we have a mapping use it if not keep the current value 
df[col] = mapped.fillna(df[col])

#check the result
df[col].value_counts(dropna=False)

In which country are you located?
The Netherlands    267
Germany            135
Spain               98
Greece              64
Austria              4
United Kingdom       4
Malta                3
France               3
Sweden               3
United States        2
Canada               2
Belgium              2
Morocco              2
Portugal             2
New Zealand          1
Singapore            1
South Africa         1
Poland               1
Italy                1
Hong Kong            1
Spiliotopoulos       1
Jordan               1
Brazil               1
Egypt                1
Turkey               1
NaN                  1
Name: count, dtype: int64

Cleaning: "What is your country of origin?"

**origin = the person's OWN country** (where they were born / live). Heritage/ancestry is NOT origin — it is already captured by the parents column and used separately in the MG1 logic. Putting heritage into origin would count it twice and wrongly turn 2nd-generation people into 1st-generation.

Judgment-call defaults:
- **Heritage answers** (born/living in country X but family from Y, e.g. "I was born in Germany, but my family is from Sri Lanka", or answers that only mention the parents) -> origin = the country where they live. The parents column then decides 2nd-gen vs non-migrant.
- **Actual migrants** (they name a foreign country they came from and now live elsewhere, e.g. "England/Spain" living in NL) -> origin = that foreign country, so origin ≠ located -> 1st generation.
- **Both-EU multi-country** (e.g. "Netherlands/Greece") -> first-listed country.
- Junk / uninterpretable answers -> Unknown (NaN)

In [8]:
#(Idrees) the second column we are cleaning
col_o = 'What is your country of origin?'


_raw_origin = pd.read_excel('Responses.xlsx')[['ID', col_o]].rename(columns={col_o: '_raw_origin'})
df = df.drop(columns=['_raw_origin'], errors='ignore').merge(_raw_origin, on='ID', how='left')
df[col_o] = df['_raw_origin']
df = df.drop(columns='_raw_origin')

#check the raw values + missing
print('Unique values:', df[col_o].nunique(dropna=True))
print('Missing:', df[col_o].isna().sum())
df[col_o].value_counts(dropna=False)

Unique values: 111
Missing: 1


What is your country of origin?
Germany        112
Spain           94
Greece          57
Netherlands     42
Germany         36
              ... 
Lebanon          1
Iran             1
Fryslan          1
Netherland       1
Amsterdam        1
Name: count, Length: 112, dtype: int64

In [9]:
#(Idrees) map every variant (case/spacing/typo/language/adjective) to one clean country name
#
#DONE (was TODO): handling people who wrote TWO/MORE countries or a heritage statement.
#KEY POINT: "origin" = the person's OWN country (where they were born / live). Heritage/ancestry is
#NOT origin - it is already captured by the parents column and used separately in the MG1 logic below.
#If we put heritage into origin we count it twice and wrongly turn 2nd-gen people into 1st-gen.
#So:
#  - Heritage answers (born/living in country X but family from Y, or only mentioning parents)
#      -> origin = the country where they live (located). The parents column then decides 2nd-gen vs non-migrant.
#  - People who actually migrated (name a foreign country they came from, now living elsewhere)
#      -> origin = that foreign country, so origin != located -> 1st generation.
origin_map = {
    #cities/region + adjective + typos
    'netherlands':'The Netherlands','the netherlands':'The Netherlands','nederland':'The Netherlands',
    'nederlands':'The Netherlands','netherland':'The Netherlands','nehterlands':'The Netherlands',
    'holland':'The Netherlands','nl':'The Netherlands','amsterdam':'The Netherlands',
    'fryslan':'The Netherlands','dutch':'The Netherlands',

    'germany':'Germany','deutschland':'Germany','german':'Germany','getmany':'Germany','germsny':'Germany',

    'spain':'Spain','españa':'Spain','espana':'Spain',

    'greece':'Greece','greece.':'Greece','greek':'Greece','creta island in greece':'Greece',

    'syria':'Syria','syrien':'Syria','سورية':'Syria',

    'morocco':'Morocco','marocco':'Morocco','morrocco':'Morocco','morroco':'Morocco',

    'turkiye':'Turkey','turkey':'Turkey','egypt':'Egypt',
    'united kingdom':'United Kingdom','uk':'United Kingdom','england':'United Kingdom',
    'usa':'United States','united states':'United States','palestine':'Palestine',

    'suriname':'Suriname','surinam':'Suriname','china':'China','chinese':'China',
    'bulgarian':'Bulgaria','irland':'Ireland','kazahstan':'Kazakhstan',
    'france':'France','frankrijk':'France',

    'india':'India','lebanon':'Lebanon','ghana':'Ghana','italy':'Italy','pakistan':'Pakistan',
    'portugal':'Portugal','eritrea':'Eritrea','vietnam':'Vietnam','taiwan':'Taiwan','canada':'Canada',
    'romania':'Romania','colombia':'Colombia','brazil':'Brazil','indonesia':'Indonesia','ecuador':'Ecuador',
    'nigeria':'Nigeria','moldova':'Moldova','aruba':'Aruba','peru':'Peru','iran':'Iran',
    'philippines':'Philippines','iraq':'Iraq','nepal':'Nepal','russia':'Russia','austria':'Austria',
    'luxembourg':'Luxembourg','argentina':'Argentina','south africa':'South Africa','latvia':'Latvia',
    'ukraine':'Ukraine','venezuela':'Venezuela','jamaica':'Jamaica','lithuania':'Lithuania','sri lanka':'Sri Lanka',

    #multi-country answers where the person names their OWN country of origin (not heritage):
    'england/ spain':'United Kingdom',     #lives in NL, came from the UK -> origin UK, 1st gen
    'netherlands/ spain':'The Netherlands',#lives in NL, both parents EU -> origin NL
    'netherlands/greece':'The Netherlands',#lives in NL, both parents EU -> origin NL
    'born in syria but originally im palestinian.':'Syria',  #born in Syria, migrated to NL -> origin Syria, 1st gen
    'suriname, molukken, irak':'The Netherlands',            #lives in NL, both parents born/brought up in NL -> origin NL
}

#heritage answers: the person was born/lives in their country of residence but has foreign family roots
#(or only describes their parents). Heritage belongs in the parents column, NOT origin -> origin = located,
#so the parents column alone decides 2nd-gen vs non-migrant.
heritage_answers = [
    'netherlands, ethnically from lebanon',                              #lives NL, family Lebanon
    'ntherlands/surinam',                                               #lives NL, family Suriname
    'dutch,surinamese & indonesian',                                    #lives NL, family Surinamese/Indonesian
    'egypt/germany',                                                    #lives Germany, Egyptian parent
    'i was born in germany, but my family is originally from sri lanka.',#born+lives Germany, family Sri Lanka
    'my parents are from pakistan',                                     #lives NL, only states parents (Pakistan)
]

normalized = df[col_o].astype(str).str.strip().str.lower()
#where we have a mapping use it; otherwise keep the current value
df[col_o] = normalized.map(origin_map).fillna(df[col_o])

#heritage answers -> origin = country of residence (df[col] = cleaned "located"); parents column decides generation
heritage_rows = normalized.isin(heritage_answers)
df.loc[heritage_rows, col_o] = df.loc[heritage_rows, col]

#junk / uninterpretable free-text -> Unknown (NaN)
junk = ['yes', 'manerf']
df.loc[df[col_o].astype(str).str.strip().str.lower().isin(junk), col_o] = np.nan

#check the result
df[col_o].value_counts(dropna=False)

What is your country of origin?
Germany            162
The Netherlands    129
Spain              120
Greece              79
Syria               23
Morocco             12
Suriname             6
India                4
Portugal             4
Colombia             3
China                3
Egypt                3
United Kingdom       3
NaN                  3
Nigeria              3
Taiwan               3
Ukraine              2
Vietnam              2
Indonesia            2
Palestine            2
Turkey               2
Pakistan             2
Romania              2
Brazil               2
Lebanon              2
Eritrea              2
France               2
Ghana                2
United States        1
Latvia               1
South Africa         1
Canada               1
Ireland              1
Luxembourg           1
Bulgaria             1
Italy                1
Austria              1
Lithuania            1
Jamaica              1
Russia               1
Aruba                1
Moldova              1
Ec

#(Idrees) build the single migrant-generation variable (MG1) from the 3 cleaned columns 
-First generation: country of origin differs from country of residence (the person migrated)
-Second generation: same country, but at least one parent is from outside the EU (the parents migrated)
-Non-migrant: same country and both parents from the EU
-Unknown: country of origin or residence is missing

In [10]:
#(Idrees) FIX 1: the 'origin_*' one-hot columns don't exist in Medeea's file,
#so nonEU_parent was always False -> 0 second-generation people.
#Now read the parents TEXT column directly: any answer with "outside" = at least one non-EU parent.
parents_col = [c for c in df.columns if c.startswith('Which description fits your parents')][0]

origin  = df[col_o]   #cleaned country of origin
located = df[col]     #cleaned country of residence

nonEU_parent = df[parents_col].astype(str).str.lower().str.contains('outside')
same    = origin.eq(located)
missing = origin.isna() | located.isna()

df['MG1'] = np.select(
    [missing, ~same, same & nonEU_parent],
    ['Unknown', 'First generation', 'Second generation'],
    default='Non-migrant',
)
df['MG1'].value_counts(dropna=False)
#expected: Non-migrant ~330 | First generation ~203 | Second generation ~67 | Unknown 3

MG1
Non-migrant          330
First generation     203
Second generation     67
Unknown                3
Name: count, dtype: int64

## ON1,ON2,ON3

In [11]:
#(Idrees) FIX 2: never flip a column that was saved already flipped.
#Rebuild ON1-ON3 from the RAW file every run, so re-running always gives the same result.
likert = {'Strongly disagree':1,'Disagree':2,'Neutral':3,'Agree':4,'Strongly agree':5}

raw = pd.read_excel('Responses.xlsx')
raw.columns = raw.columns.str.replace(r'\s+', ' ', regex=True).str.strip()

#STEP 1 (run once): find the exact ON question texts in the raw file
#for c in raw.columns: print(c)

#STEP 2: a unique keyword from each ON question (lowercase)
on_key = {
    'ON1': 'good mix of topics to stay informed',   #"my feed keeps me informed"
    'ON2': 'hides certain social or political views', #the "algorithm hides views" question
    'ON3': 'fair chance to be seen',                  #"i get a fair chance to be seen"
}

tmp = raw[['ID']].copy()
for code, key in on_key.items():
    hits = [c for c in raw.columns if key in c.lower()]
    assert len(hits) == 1, f'{code}: keyword matched {len(hits)} columns -> make it more specific'
    tmp[code] = pd.to_numeric(raw[hits[0]].replace(likert), errors='coerce')

df = df.drop(columns=['ON1','ON2','ON3'], errors='ignore').merge(tmp, on='ID', how='left')

assert df['ON2'].mean() > 3, 'ON2 is not raw - it was already flipped somewhere'
df['ON2'] = 6 - df['ON2']   #reverse ON2 so high = good, like ON1/ON3

print(df[['ON1','ON2','ON3']].describe().round(2))

          ON1     ON2     ON3
count  598.00  597.00  596.00
mean     3.33    2.31    3.09
std      0.97    0.91    0.81
min      1.00    1.00    1.00
25%      3.00    2.00    3.00
50%      3.00    2.00    3.00
75%      4.00    3.00    4.00
max      5.00    5.00    5.00


Why we reverse ON2? 
think of it like a thumbs up score
ON1: “My feed keeps me informed” -> 5 = thumbs up
ON3: “I get a fair chance to be seen” -> 5 = thumbs up 
ON2: “The algorithm hides views” -> 5 = thumbs down

ON2 is the odd one out, a high score there means something bad.
so we flip it so that all 3 

## DVP1–DVP5  (DVP2 & DVP5 added; all reverse-coded except DVP1)

In [12]:
#(Idrees) agree-disagree items -> 1-5, reverse DVP3 and DVP4 so a HIGH score = more diverse viewpoints
#(DVP2 and DVP5 use different response scales and are handled in the next cell)
dvp_cols = {
    'DVP1': 'When I post about topics like politics or social issues, it feels like my voice can be seen and heard',
    'DVP3': 'I feel like the algorithm shapes what political topics I get to see',
    'DVP4': 'I rarely see posts that challenge my own political views unless I search for them',
}
for code, c in dvp_cols.items():
    df[code] = pd.to_numeric(df[c].replace(likert), errors='coerce')

df['DVP3'] = 6 - df['DVP3']
df['DVP4'] = 6 - df['DVP4']


print(df[['DVP1','DVP3','DVP4']].corr().round(2))

      DVP1  DVP3  DVP4
DVP1  1.00 -0.05 -0.13
DVP3 -0.05  1.00  0.22
DVP4 -0.13  0.22  1.00


In [13]:
#(Idrees) FIX 3: DVP2 ("seen biased content") and DVP5 ("seen shadowbanning") are BAD experiences,
#so with our direction (high = more diverse viewpoints = good) they must be reversed, like DVP3/DVP4.
#Without this the DVP items cancel each other out (alpha was -0.19).
#Safe on re-run: both are rebuilt from the text columns first, then flipped once.
dvp2_map = {'Never':1,'Rarely':2,'Sometimes':3,'Often':4,'Very often':5}
dvp5_map = {'No, never':1, "I've heard of it but haven't seen it":2,
            'Yes, a few times':3, 'Yes, many times':4}

df['DVP2'] = df['I have seen political content that felt one-sided or biased'].map(dvp2_map)
df['DVP5'] = df[[c for c in df.columns if c.startswith("I've seen creators mention")][0]].map(dvp5_map)

df['DVP2'] = 6 - df['DVP2']   #5-point scale -> 6 - x
df['DVP5'] = 5 - df['DVP5']   #4-point scale -> 5 - x

#z-scores so the 1-5 and 1-4 scales count equally, then average all 5 items
dvp_items = ['DVP1','DVP2','DVP3','DVP4','DVP5']
for c in dvp_items:
    df[c+'_z'] = (df[c] - df[c].mean()) / df[c].std()
df['DVP_score'] = df[[c+'_z' for c in dvp_items]].mean(axis=1)

print(df[dvp_items].corr().round(2))   #DVP2-DVP5 should now mostly be positive

      DVP1  DVP2  DVP3  DVP4  DVP5
DVP1  1.00 -0.02 -0.05 -0.13 -0.04
DVP2 -0.02  1.00  0.37  0.11  0.28
DVP3 -0.05  0.37  1.00  0.22  0.16
DVP4 -0.13  0.11  0.22  1.00  0.06
DVP5 -0.04  0.28  0.16  0.06  1.00


In [14]:
#(Idrees) frequency checking: any item where almost everyone gives the same answer?
for c in ['ON1','ON2','ON3','DVP1','DVP2','DVP3','DVP4','DVP5']:
    print(c, df[c].value_counts(normalize=True).sort_index().round(2).to_dict())

ON1 {1.0: 0.03, 2.0: 0.19, 3.0: 0.28, 4.0: 0.42, 5.0: 0.08}
ON2 {1.0: 0.17, 2.0: 0.46, 3.0: 0.26, 4.0: 0.09, 5.0: 0.01}
ON3 {1.0: 0.03, 2.0: 0.19, 3.0: 0.48, 4.0: 0.28, 5.0: 0.03}
DVP1 {1.0: 0.06, 2.0: 0.17, 3.0: 0.47, 4.0: 0.27, 5.0: 0.04}
DVP2 {1.0: 0.31, 2.0: 0.39, 3.0: 0.26, 4.0: 0.04, 5.0: 0.01}
DVP3 {1.0: 0.22, 2.0: 0.51, 3.0: 0.2, 4.0: 0.06, 5.0: 0.01}
DVP4 {1.0: 0.08, 2.0: 0.36, 3.0: 0.29, 4.0: 0.24, 5.0: 0.03}
DVP5 {1.0: 0.27, 2.0: 0.4, 3.0: 0.23, 4.0: 0.1}


# PF  (perceived fairness)
-F1: "I trust the algorithm to recommend me content fairly" (trust in algorithmic fairness)
-F2: "I think social media is fair2" (general fairness, end of survey)

PF = the mean of F1 and F2, which keeps it on the original 1–5 scale (comparable to the ON/DVP items). We also correlate F1 & F2 to check they move together before averaging.

In [15]:
#(Idrees)- 
f_cols = {
    'F1': 'I trust the algorithm to recommend me content fairly',
    'F2': 'I think social media is fair2',
}
for code, c in f_cols.items():
    df[code] = pd.to_numeric(df[c].replace(likert), errors='coerce')


df['PF'] = df[['F1','F2']].mean(axis=1)

print('PF mean (overall):', round(df['PF'].mean(), 2))
print(df[['F1','F2','PF']].describe().round(2))

print('\nF1-F2 correlation:')
print(df[['F1','F2']].corr().round(2))

PF mean (overall): 2.75
           F1      F2      PF
count  596.00  597.00  600.00
mean     2.85    2.66    2.75
std      0.95    0.92    0.80
min      1.00    1.00    1.00
25%      2.00    2.00    2.00
50%      3.00    3.00    3.00
75%      4.00    3.00    3.00
max      5.00    5.00    5.00

F1-F2 correlation:
      F1    F2
F1  1.00  0.48
F2  0.48  1.00


## BC1–BC5  (Biased Content)
Paper direction: **high = more perceived bias.** BC2 & BC3 ("equal views") are reverse-coded so agreeing (fair) → low. Built from the raw question text in `Responses.xlsx` (the working file only has Mats' pre-coded BC, which uses the opposite direction).

In [16]:
#(Idrees) BC = Biased Content, paper direction: high = more perceived bias
#the working file only holds Mats' pre-coded BC1-BC5 (his direction = high FAIR), so to build my OWN BC
#(high = bias) I pull the RAW question text from Responses.xlsx and overwrite BC1-BC5 with my coding.
bc_cols = {
    'BC1': 'My feed does not show information others are seeing',
    'BC2': 'My content gets the same number of views as content from others',
    'BC3': 'My content gets as many views as others from different backgrounds',
    'BC4': 'I think content from certain groups of people is prioritized over other groups',
    'BC5': 'I am informed that my feed is shaped by personal information like my age, gender or location',
}
#distinctive keyword(s) to find each question in the raw file (the real wording differs from the paper
#labels above and has stray non-breaking/double spaces, so we match on keywords, not the full text)
bc_key = {
    'BC1': ['feed does not show information that others'],
    'BC2': ['gets the same number of views as content from others'],   # plain one (no "different background")
    'BC3': ['gets the same number of views', 'different background'],   # the different-background one
    'BC4': ['certain groups of people is prioritized'],
    'BC5': ['i am informed that my feed is shaped'],
}
raw = pd.read_excel('Responses.xlsx')
raw.columns = raw.columns.str.replace(r'\s+', ' ', regex=True).str.strip()   # kill nbsp / double spaces

tmp = raw[['ID']].copy()
for code, keys in bc_key.items():
    hits = [c for c in raw.columns if all(k in c.lower() for k in keys)]
    if 'different background' not in keys:
        hits = [h for h in hits if 'different background' not in h.lower()]   # keep BC2 separate from BC3
    tmp[code] = pd.to_numeric(raw[hits[0]].replace(likert), errors='coerce')

#overwrite any existing BC columns with my own coding
df = df.drop(columns=list(bc_cols), errors='ignore').merge(tmp, on='ID', how='left')

df['BC2'] = 6 - df['BC2']   # "equal views" -> flip so high = bias
df['BC3'] = 6 - df['BC3']
print(df[list(bc_cols)].corr().round(2))

      BC1   BC2   BC3   BC4   BC5
BC1  1.00 -0.07  0.03  0.05 -0.03
BC2 -0.07  1.00  0.46  0.14  0.09
BC3  0.03  0.46  1.00  0.13  0.02
BC4  0.05  0.14  0.13  1.00  0.13
BC5 -0.03  0.09  0.02  0.13  1.00


In [17]:
#save to Medeea's final combined file
df.to_excel('Responses_cleaned_finalMedeea.xlsx', index=False)
print('saved in Responses_cleaned_finalMedeea.xlsx')

saved in Responses_cleaned_finalMedeea.xlsx
